# 01 — Explore & describe: NYC 311 (2025 subset)

Loads the two raw 311 windows (winter + summer 2025) as downloaded by `00_download.ipynb` and
gets a first, purely descriptive look: shape, date coverage, missing values, top complaint types,
borough counts, duplicate keys, and a full `ydata-profiling` report.

This notebook only **describes** — no cleaning or fixing here (that is Phase 3/4). Missing %
is computed on the full data, not a sample; only the profiling report itself runs on a sample
(100,000 rows) because it is otherwise slow on 1.2M+ rows.

In [1]:
import sys
import pandas as pd

print("python", sys.version)
print("pandas", pd.__version__)

python 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
pandas 2.3.3


## Load

In [2]:
df_winter = pd.read_csv("../data/raw/311_2025-01_2025-02_downloaded_2026-07-12.csv", low_memory=False)
df_summer = pd.read_csv("../data/raw/311_2025-07_2025-08_downloaded_2026-07-12.csv", low_memory=False)
df = pd.concat([df_winter, df_summer], ignore_index=True)

print("winter shape", df_winter.shape)
print("summer shape", df_summer.shape)
print("combined shape", df.shape)

winter shape (603544, 12)
summer shape (619913, 12)
combined shape (1223457, 12)


## Date coverage

In [3]:
created = pd.to_datetime(df["created_date"], errors="coerce")
print("created_date min", created.min())
print("created_date max", created.max())
print("unparseable created_date:", created.isna().sum())

created_date min 2025-01-01 00:00:12
created_date max 2025-08-31 23:59:56
unparseable created_date: 0


## Missing values per column (full data)

In [4]:
missing_pct = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
print(missing_pct)

descriptor        2.60
closed_date       1.49
latitude          1.34
longitude         1.34
incident_zip      0.78
agency            0.00
unique_key        0.00
created_date      0.00
borough           0.00
complaint_type    0.00
agency_name       0.00
status            0.00
dtype: float64


## Top 30 complaint types

In [5]:
top30 = df["complaint_type"].value_counts().head(30)
print(top30)
print()
print("total distinct complaint_type values:", df["complaint_type"].nunique())

complaint_type
Illegal Parking                  179650
Noise - Residential              157197
HEAT/HOT WATER                   113865
Noise - Street/Sidewalk           56130
Blocked Driveway                  53860
UNSANITARY CONDITION              40288
Water System                      34151
Abandoned Vehicle                 23434
PLUMBING                          23295
Street Condition                  22825
Dirty Condition                   21060
PAINT/PLASTER                     20068
Noise - Commercial                18216
Traffic Signal Condition          17646
Noise                             16529
DOOR/WINDOW                       15412
Drug Activity                     14882
Derelict Vehicles                 14773
Encampment                        14766
Missed Collection                 13662
WATER LEAK                        13190
Illegal Dumping                   13094
Homeless Person Assistance        12033
Noise - Vehicle                   11754
General Construction/Plum

total distinct complaint_type values: 180


## Borough counts

In [6]:
print(df["borough"].value_counts(dropna=False))

borough
BROOKLYN         350548
BRONX            298456
QUEENS           295369
MANHATTAN        233481
STATEN ISLAND     44792
Unspecified         811
Name: count, dtype: int64


## Duplicate `unique_key`

In [7]:
dup_count = df["unique_key"].duplicated().sum()
print("duplicated unique_key rows:", dup_count)

duplicated unique_key rows: 0


## ydata-profiling report (sample, 100,000 rows)

Full missing-value/duplicate numbers above were computed on the full 1.2M-row data; this HTML
report is a show-piece for the exam (screen-shareable) and runs on a random sample for speed.

In [8]:
from ydata_profiling import ProfileReport

sample = df.sample(min(100_000, len(df)), random_state=42)
profile = ProfileReport(sample, title="NYC 311 (2025 subset) — profiling report (100k sample)", minimal=True)
profile.to_file("../docs/311_profiling_report.html")
print("saved ../docs/311_profiling_report.html")

C:\Users\kemki\AppData\Local\Temp\ipykernel_6876\2292890232.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  8%|▊         | 1/12 [00:00<00:07,  1.44it/s]

 17%|█▋        | 2/12 [00:01<00:08,  1.23it/s]

...

100%|██████████| 12/12 [00:01<00:00,  6.47it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

saved ../docs/311_profiling_report.html


## Notes for later phases

- Casing pattern observed: HPD-style complaint types are ALL CAPS (`HEAT/HOT WATER`,
  `UNSANITARY CONDITION`, ...) while DSNY/DOT/Parks-style types are Title Case (`Dirty Condition`,
  `Street Condition`, ...) — a consistency point to revisit in Phase 3.
- Borough field includes an `Unspecified` value, which is a disguised-missing case, not a real
  category — flagged for the completeness/consistency checks in Phase 3, not fixed here.
- Full quality assessment (completeness, consistency, validity, ...) happens in
  `03_quality.ipynb`; this notebook stays descriptive only.